# Ejercicio 3: Modelo Vectorial y TF-IDF

**Realizado por:** Correa Adrian  
**Fecha:** 03/05/2026

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz **TF-IDF** para un corpus pequeño (1 libro de ejemplo).
- Calcular la matriz **TF-IDF** para el corpus Gutenberg 1000 (1000 libros).
- Implementar una función de búsqueda basada en **similitud coseno**.

---

## Teoría - Fundamentos del Modelo Vectorial

### 1. Representación Vectorial
Cada documento y consulta se representa como un vector: 
$$\vec{d}, \vec{q} \in \mathbb{R}^{n}$$
donde $n$ es el número de términos distintos en el vocabulario.

### 2. Similitud del Coseno
Medida de similitud entre consulta y documento:
$$\text{sim}(\vec{q}, \vec{d}) = \frac{\vec{q} \cdot \vec{d}}{||\vec{q}|| \cdot ||\vec{d}||}$$

### 3. Componentes TF-IDF

**Frecuencia de Términos (TF):**
$$\text{tf}_{t,d} = 1 + \log(f_{t,d})$$
donde $f_{t,d}$ es la frecuencia cruda del término $t$ en el documento $d$.

**Frecuencia Inversa de Documentos (IDF):**
$$\text{idf}_{t} = \log\left(\frac{N}{df_{t}}\right)$$
donde $N$ es el número total de documentos y $df_t$ es el número de documentos que contienen el término $t$.

**Ponderación TF-IDF:**
$$w_{t,d} = \text{tf}_{t,d} \cdot \text{idf}_{t}$$

## FASE 1: PREPARACIÓN Y CARGA DE DATOS

### Paso 1: Calcular la matriz TF-IDF para un corpus pequeño

Como no existe el archivo `data/01_corpus_turismo_500.txt` en este repositorio, usaremos un libro en español disponible en la carpeta `Data` como corpus de ejemplo. Para este paso trabajaremos con **`Amor y Pedagogía.txt`**.

In [9]:
import os
import re
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================================
# PASO 1: TF-IDF PARA UN LIBRO EN ESPAÑOL
# ============================================================================

ruta_libro = r"d:\7mo semestre\RI\books2\Data\Amor y Pedagogía.txt"

if not os.path.exists(ruta_libro):
    raise FileNotFoundError(f"No se encontró el libro en la ruta: {ruta_libro}")

with open(ruta_libro, "r", encoding="utf-8", errors="ignore") as archivo:
    texto_libro = archivo.read()

# Limpieza básica del texto
texto_libro = texto_libro.lower()
texto_libro = texto_libro.translate(str.maketrans("", "", string.punctuation))
texto_libro = re.sub(r"\d+", "", texto_libro)
texto_libro = re.sub(r"\s+", " ", texto_libro).strip()

# Para que TF-IDF tenga más de una observación, dividimos el libro en fragmentos
palabras = texto_libro.split()
ventana = 250
solapamiento = 50
fragmentos = []

for inicio in range(0, len(palabras), ventana - solapamiento):
    fragmento = " ".join(palabras[inicio:inicio + ventana]).strip()
    if len(fragmento.split()) >= 30:
        fragmentos.append(fragmento)

vectorizador_paso1 = TfidfVectorizer(
    stop_words=[
        'el', 'la', 'de', 'que', 'y', 'a', 'en', 'un', 'ser', 'se', 'no', 'haber',
        'por', 'con', 'su', 'para', 'es', 'al', 'lo', 'como', 'más', 'o', 'pero',
        'sus', 'le', 'ya', 'fue', 'este', 'ha', 'porque', 'esta', 'son', 'entre',
        'está', 'cuando', 'muy', 'sin', 'sobre', 'tiene', 'también', 'me', 'hasta',
        'hay', 'donde', 'han', 'quien', 'están', 'estado', 'desde', 'todo', 'nos',
        'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'fueron', 'ese',
        'eso', 'había', 'ante', 'ellos', 'esto', 'antes'
    ],
    token_pattern=r'(?u)\b[a-záéíóúñ]{3,}\b',
    min_df=2,
    max_df=0.9,
    norm='l2',
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True
)

# Ajuste para que IDF coincida con la fórmula de las diapositivas: idf_t = log(N/df_t)
vectorizador_paso1.set_params(smooth_idf=False, use_idf=True)
vectorizador_paso1.fit(fragmentos)

# Calcular df (número de documentos que contienen cada término)
tfidf_counts = (vectorizador_paso1.transform(fragmentos) > 0).astype(int)
df_counts = np.asarray(tfidf_counts.sum(axis=0)).ravel()
N_docs = len(fragmentos)
idf_target = np.log(N_docs / df_counts)

# Sobrescribir idf en el vectorizador y su transformador interno
vectorizador_paso1.idf_ = idf_target
if hasattr(vectorizador_paso1, '_tfidf'):
    vectorizador_paso1._tfidf.idf_ = idf_target

# Volver a transformar usando la idf explícita
matriz_tfidf_paso1 = vectorizador_paso1.transform(fragmentos)
vocabulario_paso1 = vectorizador_paso1.get_feature_names_out()

print("\n" + "=" * 80)
print("PASO 1: MATRIZ TF-IDF USANDO 'AMOR Y PEDAGOGÍA'")
print("=" * 80)
print(f"Libro utilizado: {os.path.basename(ruta_libro)}")
print(f"Cantidad de fragmentos generados: {len(fragmentos)}")
print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_paso1.shape[0]} documentos × {matriz_tfidf_paso1.shape[1]} términos")
print(f"Elementos no-cero: {matriz_tfidf_paso1.nnz}")

print("\nPrimeros 25 términos del vocabulario:")
print(", ".join(vocabulario_paso1[:25]))

# Mostrar una vista pequeña de la matriz como DataFrame
vista = pd.DataFrame(
    matriz_tfidf_paso1[:5].toarray(),
    columns=vocabulario_paso1
)
print("\nVista parcial de la matriz TF-IDF (primeros 5 fragmentos):")
print(vista.iloc[:, :15].head())


PASO 1: MATRIZ TF-IDF USANDO 'AMOR Y PEDAGOGÍA'
Libro utilizado: Amor y Pedagogía.txt
Cantidad de fragmentos generados: 244
Dimensiones de la matriz TF-IDF: 244 documentos × 4008 términos
Elementos no-cero: 22391

Primeros 25 términos del vocabulario:
abajo, abalanzarse, abceso, abejas, aberraciones, abierta, abierto, abiertos, abismo, abismos, aborrece, aborrezca, abortado, abraza, abrazado, abrazan, abre, abrirle, absoluta, absolutamente, absoluto, abstracto, absurda, absurdas, absurdo

Vista parcial de la matriz TF-IDF (primeros 5 fragmentos):
   abajo  abalanzarse  abceso  abejas  aberraciones  abierta  abierto  \
0    0.0          0.0     0.0     0.0      0.000000      0.0      0.0   
1    0.0          0.0     0.0     0.0      0.000000      0.0      0.0   
2    0.0          0.0     0.0     0.0      0.000000      0.0      0.0   
3    0.0          0.0     0.0     0.0      0.000000      0.0      0.0   
4    0.0          0.0     0.0     0.0      0.133423      0.0      0.0   

   abie

### Paso 2: Construir el corpus Gutenberg 1000

**INFORMACIÓN DE ARCHIVOS:**
- **Ruta:** `D:\7mo semestre\RI\books2\Data\`
- **Cantidad de archivos:** 1000 archivos `.txt` individuales (tomamos los primeros 1000 en orden alfabético)
- **Tipo:** Libros de Project Gutenberg (dominio público)

In [5]:
import glob
import time

print("\n" + "=" * 80)
print("PASO 2: CONSTRUIR EL CORPUS GUTENBERG 1000")
print("=" * 80)

print("\nINFORMACIÓN DE ARCHIVOS:")
print("-" * 80)
print("Ruta:                              D:\\7mo semestre\\RI\\books2\\Data\\")
print("Tipo de archivos:                  .txt individuales (1 libro por archivo)")
print("Cantidad total de archivos:        1000 (primeros 1000 en orden alfabético)")
print("-" * 80 + "\n")

ruta_corpus = r"d:\7mo semestre\RI\books2\Data"
archivos_todos = sorted(glob.glob(os.path.join(ruta_corpus, "*.txt")))
archivos_1000 = archivos_todos[:1000]

print(f"Total de archivos encontrados en Data: {len(archivos_todos)}")
print(f"Archivos que se usarán en el corpus Gutenberg 1000: {len(archivos_1000)}")
print("\nPrimeros 10 archivos:")
for indice, archivo in enumerate(archivos_1000[:10], 1):
    print(f"  {indice:2d}. {os.path.basename(archivo)}")
print("  ...\n")


def limpiar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans("", "", string.punctuation))
    texto = re.sub(r"\d+", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


stop_words_es = [
    "el", "la", "de", "que", "y", "a", "en", "un", "ser", "se", "no", "haber",
    "por", "con", "su", "para", "es", "al", "lo", "como", "más", "o", "pero",
    "sus", "le", "ya", "fue", "este", "ha", "porque", "esta", "son", "entre",
    "está", "cuando", "muy", "sin", "sobre", "tiene", "también", "me", "hasta",
    "hay", "donde", "han", "quien", "están", "estado", "desde", "todo", "nos",
    "durante", "todos", "uno", "les", "ni", "contra", "otros", "fueron", "ese",
    "eso", "había", "ante", "ellos", "esto", "antes"
]

print("Cargando documentos...")
tiempo_inicio = time.time()
documentos_1000 = []
nombres_docs_1000 = []

for indice, archivo in enumerate(archivos_1000, 1):
    try:
        with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
            contenido = f.read()
        texto_limpio = limpiar_texto(contenido[:25000])
        if len(texto_limpio) > 100:
            documentos_1000.append(texto_limpio)
            nombres_docs_1000.append(os.path.basename(archivo))
        if len(documentos_1000) % 250 == 0:
            print(f"  ✓ {len(documentos_1000)} documentos procesados")
    except Exception:
        pass

tiempo_carga_1000 = time.time() - tiempo_inicio

print(f"\n{'=' * 80}")
print("ESTADÍSTICAS DEL CORPUS GUTENBERG 1000")
print(f"{'=' * 80}")
print(f"  Total de documentos cargados: {len(documentos_1000)}")
print(f"  Tiempo de carga: {tiempo_carga_1000:.2f}s")
palabras_por_doc = [len(doc.split()) for doc in documentos_1000]
print(f"  Palabras promedio/documento: {int(np.mean(palabras_por_doc))}")
print(f"  Palabras min/max: {int(np.min(palabras_por_doc))} / {int(np.max(palabras_por_doc))}")


PASO 2: CONSTRUIR EL CORPUS GUTENBERG 1000

INFORMACIÓN DE ARCHIVOS:
--------------------------------------------------------------------------------
Ruta:                              D:\7mo semestre\RI\books2\Data\
Tipo de archivos:                  .txt individuales (1 libro por archivo)
Cantidad total de archivos:        1000 (primeros 1000 en orden alfabético)
--------------------------------------------------------------------------------

Total de archivos encontrados en Data: 1000
Archivos que se usarán en el corpus Gutenberg 1000: 1000

Primeros 10 archivos:
   1. 20 poemas para ser leídos en el tranvía.txt
   2. 40 years  40 años  40 ans.txt
   3. 7 de julio.txt
   4. A Doll's House  a play.txt
   5. A First Spanish Reader.txt
   6. A Flor De Piel Frases.txt
   7. A History of Magic and Experimental Science, Volume 1 (of 2) During the First Thirteen Centuries of .txt
   8. A Modest Proposal For preventing the children of poor people in Ireland, from being a burden on thei.tx

### Paso 3: Calcular la matriz TF-IDF para el corpus Gutenberg 1000

In [10]:
print("\n" + "=" * 80)
print("PASO 3: CALCULAR LA MATRIZ TF-IDF PARA EL CORPUS GUTENBERG 1000")
print("=" * 80)

vectorizador = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_es,
    token_pattern=r"(?u)\b[a-záéíóúñ]{3,}\b",
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    norm="l2",
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=True,
)

tiempo_inicio = time.time()
# Ajuste para que IDF coincida con la diapositiva: idf = log(N/df)
vectorizador.fit(documentos_1000)

# Calcular df (número de documentos que contienen cada término)
tfidf_counts_1000 = (vectorizador.transform(documentos_1000) > 0).astype(int)
df_counts_1000 = np.asarray(tfidf_counts_1000.sum(axis=0)).ravel()
N_docs_1000 = len(documentos_1000)
idf_target_1000 = np.log(N_docs_1000 / df_counts_1000)

# Sobrescribir idf en el vectorizador y su transformador interno
vectorizador.idf_ = idf_target_1000
if hasattr(vectorizador, '_tfidf'):
    vectorizador._tfidf.idf_ = idf_target_1000

# Transformar con la idf personalizada
matriz_tfidf = vectorizador.transform(documentos_1000)
tiempo_vec = time.time() - tiempo_inicio

print(f"\n{'=' * 80}")
print("MATRIZ TF-IDF GENERADA")
print(f"{'=' * 80}")
print(f"  Dimensiones: {matriz_tfidf.shape[0]} documentos × {matriz_tfidf.shape[1]} términos")
print(f"  Elementos no-cero: {matriz_tfidf.nnz:,}")
densidad = (matriz_tfidf.nnz / (matriz_tfidf.shape[0] * matriz_tfidf.shape[1])) * 100
print(f"  Densidad: {densidad:.4f}%")
print(f"  Tiempo de vectorización: {tiempo_vec:.2f}s")

nombres_features = vectorizador.get_feature_names_out()
print(f"\n  Vocabulario total: {len(nombres_features)} términos únicos")
print(f"  Primeros 30 términos: {', '.join(nombres_features[:30])}")


PASO 3: CALCULAR LA MATRIZ TF-IDF PARA EL CORPUS GUTENBERG 1000

MATRIZ TF-IDF GENERADA
  Dimensiones: 1000 documentos × 5000 términos
  Elementos no-cero: 689,774
  Densidad: 13.7955%
  Tiempo de vectorización: 26.61s

  Vocabulario total: 5000 términos únicos
  Primeros 30 términos: abad, abajo, abandonado, abandonar, abandono, abierta, abierto, abismo, able, abogado, about, about the, above, abre, abriendo, abrigo, abril, abrir, abrió, absoluta, absolutamente, absoluto, abuela, abuelo, abundancia, abundante, acaba, acababa, acabado, acabar


### Análisis: Términos más relevantes en el corpus

In [4]:
print("\n" + "="*80)
print("ANÁLISIS DE VOCABULARIO Y TÉRMINOS RELEVANTES")
print("="*80)

# Calcular TF-IDF promedio para cada término
tfidf_promedio = np.asarray(matriz_tfidf.mean(axis=0)).ravel()
indices_top = np.argsort(tfidf_promedio)[::-1][:50]

print(f"\nEstadísticas TF-IDF:")
print(f"  Mínimo: {matriz_tfidf.min():.6f}")
print(f"  Máximo: {matriz_tfidf.max():.6f}")
print(f"  Promedio: {tfidf_promedio.mean():.6f}")
print(f"  Desviación estándar: {tfidf_promedio.std():.6f}")

print(f"\nTop 30 términos por relevancia TF-IDF promedio:")
print(f"{'Rank':<5} {'Término':<25} {'TF-IDF Promedio':<20}")
print("-" * 50)

for i, idx in enumerate(indices_top[:30], 1):
    termino = nombres_features[idx]
    score = tfidf_promedio[idx]
    print(f"{i:<5} {termino:<25} {score:>18.6f}")


ANÁLISIS DE VOCABULARIO Y TÉRMINOS RELEVANTES

Estadísticas TF-IDF:
  Mínimo: 0.000000
  Máximo: 0.578897
  Promedio: 0.004745
  Desviación estándar: 0.003100

Top 30 términos por relevancia TF-IDF promedio:
Rank  Término                   TF-IDF Promedio     
--------------------------------------------------
1     era                                 0.026830
2     tan                                 0.025722
3     qué                                 0.025059
4     dos                                 0.024198
5     bien                                0.023468
6     pues                                0.022467
7     vida                                0.022255
8     mismo                               0.022027
9     ella                                0.021515
10    tiempo                              0.021307
11    así                                 0.021299
12    the                                 0.020780
13    hombre                              0.020653
14    después           

### Paso 4: Programar una función `buscar()` para el corpus Gutenberg 1000

In [11]:
print("\n" + "="*80)
print("PASO 4: FUNCIÓN DE BÚSQUEDA - SIMILITUD COSENO")
print("="*80)

def buscar(consulta, corpus_tfidf, vectorizer, nombres_documentos, top_k=10):
    """
    Busca documentos relevantes usando similitud coseno.
    
    Parámetros:
    -----------
    consulta : str
        Texto de la consulta a buscar
    corpus_tfidf : matriz sparse
        Matriz TF-IDF del corpus (n_docs × n_features)
    vectorizer : TfidfVectorizer
        Vectorizador entrenado con el corpus
    nombres_documentos : list
        Nombres/IDs de los documentos
    top_k : int
        Número de resultados a retornar
    
    Retorna:
    --------
    pd.DataFrame
        Tabla con Rank, Documento y Similitud (%)
        
    Fórmula matemática:
    -------------------
    sim(q, d) = (q · d) / (||q|| × ||d||)
    """
    
    # Paso 1: Limpiar y vectorizar la consulta
    consulta_limpia = limpiar_texto(consulta)
    vector_consulta = vectorizer.transform([consulta_limpia])
    
    # Paso 2: Calcular similitud coseno
    similitudes = cosine_similarity(vector_consulta, corpus_tfidf)[0]
    
    # Paso 3: Ordenar por relevancia (descendente)
    indices_ordenados = np.argsort(similitudes)[::-1][:top_k]
    
    # Paso 4: Construir tabla de resultados
    resultados = []
    for rank, idx in enumerate(indices_ordenados, 1):
        if similitudes[idx] > 0:
            resultados.append({
                'Rank': rank,
                'Documento': nombres_documentos[idx],
                'Similitud': similitudes[idx],
                'Similitud (%)': f"{similitudes[idx]*100:.2f}%"
            })
    
    return pd.DataFrame(resultados)


# Demostración: Ejecutar búsquedas de prueba
print("\nEjecutando búsquedas en el corpus Gutenberg 1000:\n")

consultas_demo = [
    'amor pasión romance',
    'misterio intriga secreto',
    'aventura viaje exploración',
    'muerte tragedia dolor',
    'esperanza futuro vida'
]

for i, consulta in enumerate(consultas_demo, 1):
    print(f"\n{'='*80}")
    print(f"CONSULTA {i}: \"{consulta}\"")
    print(f"{'='*80}")
    
    resultados = buscar(consulta, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=8)
    
    if len(resultados) > 0:
        print(resultados.to_string(index=False))
    else:
        print("  [Sin resultados relevantes]")
    
    print()


PASO 4: FUNCIÓN DE BÚSQUEDA - SIMILITUD COSENO

Ejecutando búsquedas en el corpus Gutenberg 1000:


CONSULTA 1: "amor pasión romance"
 Rank                                                                                       Documento  Similitud Similitud (%)
    1                                 Las cien mejores poesías (lí­ricas) de la lengua castellana.txt   0.108368        10.84%
    2                                                                            Romancero gitano.txt   0.098185         9.82%
    3                                                                          Novelas ejemplares.txt   0.097383         9.74%
    4                                                                             Libro de poemas.txt   0.092912         9.29%
    5                                                                            Novelas y teatro.txt   0.087268         8.73%
    6 Historia de la lengua y literatura castellana, Tomo 1  $b Desde los orígenes hasta Carlos V.txt  

In [13]:
print("\n" + "="*80)
print("BÚSQUEDAS AVANZADAS Y VALIDACIÓN")
print("="*80)

# Comparación: Autoconsulta (documento como consulta)
print("\nValidación 1: Autoconsulta (documento comparado consigo mismo)")
print("-" * 80)

idx_prueba = 100
doc_prueba = documentos_1000[idx_prueba]
nombre_prueba = nombres_docs_1000[idx_prueba]

vector_doc = vectorizador.transform([doc_prueba])
similitudes = cosine_similarity(vector_doc, matriz_tfidf)[0]

indices_similares = np.argsort(similitudes)[::-1][1:6]

print(f"\nDocumento de referencia: {nombre_prueba}")
print(f"Primeras palabras: {' '.join(doc_prueba.split()[:15])}...\n")
print("Documentos más similares:")

for rank, idx in enumerate(indices_similares, 1):
    print(f"  {rank}. {nombres_docs_1000[idx]:<60} -> {similitudes[idx]*100:6.2f}%")

# Comparación: Búsquedas complejas
print(f"\n{'='*80}")
print("Validación 2: Búsquedas con diferentes niveles de especificidad")
print("="*80)

consultas_variadas = [
    'libro',  # Muy general
    'príncipe reino',  # Moderada
    'príncipe dinamarca tragedia shakespeare',  # Específica
]

for consulta in consultas_variadas:
    resultados = buscar(consulta, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=3)
    print(f"\nConsulta: \"{consulta}\"")
    if len(resultados) > 0:
        similitud_max = resultados['Similitud'].max()
        print(f"  Relevancia máxima: {similitud_max*100:.2f}%")
        print(f"  Resultado top: {resultados.iloc[0]['Documento']}")
    else:
        print("  [Sin resultados]")


BÚSQUEDAS AVANZADAS Y VALIDACIÓN

Validación 1: Autoconsulta (documento comparado consigo mismo)
--------------------------------------------------------------------------------

Documento de referencia: Comentario del coronel Francisco Verdugo, de la guerra de Frisia, en xiv años que fue gobernador y c.txt
Primeras palabras: coleccion de libros españoles raros ó curiosos tomo segundo comentario del coronel francisco verdugo de...

Documentos más similares:
  1. Expedición de Catalanes y Aragoneses al Oriente.txt          ->  31.24%
  2. Historia de la Conquista de Mexico, Volume 2 (of 3) Poblacion y Progresos de la America Septentriona.txt ->  30.25%
  3. Historia de las Indias (vol. 5 de 5).txt                     ->  28.59%
  4. Verdadera historia de los sucesos de la conquista de la Nueva-España (3 de 3).txt ->  28.30%
  5. Historia de las Indias (vol. 3 de 5).txt                     ->  28.13%

Validación 2: Búsquedas con diferentes niveles de especificidad

Consulta: "libro"
  R

In [16]:
print("\n" + "="*80)
print("COMPARACIÓN: CORPUS DE 500 vs 1000 DOCUMENTOS")
print("="*80)

# Esta celda es autocontenida: crea un corpus de 500 documentos a partir
# del mismo corpus Gutenberg 1000 para evitar depender de variables antiguas.
documentos_500 = documentos_1000[:500]
nombres_docs_500 = nombres_docs_1000[:500]
vectorizador_500 = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_es,
    token_pattern=r"(?u)\b[a-záéíóúñ]{3,}\b",
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    norm="l2",
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=True,
)

vectorizador_500.fit(documentos_500)
tfidf_counts_500 = (vectorizador_500.transform(documentos_500) > 0).astype(int)
df_counts_500 = np.asarray(tfidf_counts_500.sum(axis=0)).ravel()
N_docs_500 = len(documentos_500)
idf_target_500 = np.log(N_docs_500 / df_counts_500)
vectorizador_500.idf_ = idf_target_500
if hasattr(vectorizador_500, '_tfidf'):
    vectorizador_500._tfidf.idf_ = idf_target_500
matriz_tfidf_500 = vectorizador_500.transform(documentos_500)

# Analizar diferencias en matrices
print(f"\nCaracterísticas de los dos corpus:")
print(f"{'Métrica':<30} {'500 docs':<20} {'1000 docs':<20}")
print("-" * 70)
print(f"{'Dimensión matriz':<30} {matriz_tfidf_500.shape[1]:<20} {matriz_tfidf.shape[1]:<20}")
nnz_500 = f"{matriz_tfidf_500.nnz:,}"
nnz_1000 = f"{matriz_tfidf.nnz:,}"
print(f"{'Elementos no-cero':<30} {nnz_500:<20} {nnz_1000:<20}")
densidad_500_calc = (matriz_tfidf_500.nnz / (matriz_tfidf_500.shape[0] * matriz_tfidf_500.shape[1])) * 100
densidad_1000_calc = (matriz_tfidf.nnz / (matriz_tfidf.shape[0] * matriz_tfidf.shape[1])) * 100
print(f"{'Densidad (%)':<30} {densidad_500_calc:.4f}%{'':<13} {densidad_1000_calc:.4f}%")

# Buscar la misma consulta en ambos corpus
consulta_test = 'aventura historia'

print(f"\n\nBúsqueda de \"{consulta_test}\" en ambos corpus:\n")

resultados_500 = buscar(consulta_test, matriz_tfidf_500, vectorizador_500, nombres_docs_500, top_k=5)
resultados_1000 = buscar(consulta_test, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=5)

print("En corpus de 500 documentos:")
if len(resultados_500) > 0:
    print(resultados_500[['Rank', 'Similitud (%)']].to_string(index=False))
    print(f"Relevancia promedio: {resultados_500['Similitud'].mean()*100:.2f}%")
else:
    print("  [Sin resultados]")

print(f"\nEn corpus de 1000 documentos:")
if len(resultados_1000) > 0:
    print(resultados_1000[['Rank', 'Similitud (%)']].to_string(index=False))
    print(f"Relevancia promedio: {resultados_1000['Similitud'].mean()*100:.2f}%")
else:
    print("  [Sin resultados]")


COMPARACIÓN: CORPUS DE 500 vs 1000 DOCUMENTOS

Características de los dos corpus:
Métrica                        500 docs             1000 docs           
----------------------------------------------------------------------
Dimensión matriz               5000                 5000                
Elementos no-cero              353,553              689,774             
Densidad (%)                   14.1421%              13.7955%


Búsqueda de "aventura historia" en ambos corpus:

En corpus de 500 documentos:
 Rank Similitud (%)
    1        13.98%
    2         9.90%
    3         8.03%
    4         6.49%
    5         6.32%
Relevancia promedio: 8.94%

En corpus de 1000 documentos:
 Rank Similitud (%)
    1        14.97%
    2        10.89%
    3         8.42%
    4         8.18%
    5         7.75%
Relevancia promedio: 10.04%


In [17]:
# Compact verification of Paso 4: show top-8 results for each demo query
consultas_compactas = [
    'amor pasión romance',
    'misterio intriga secreto',
    'aventura viaje exploración',
    'muerte tragedia dolor',
    'esperanza futuro vida'
]

for consulta in consultas_compactas:
    print('\n' + '='*80)
    print(f'CONSULTA: "{consulta}"')
    print('='*80)
    resultados = buscar(consulta, matriz_tfidf, vectorizador, nombres_docs_1000, top_k=8)
    if resultados.empty:
        print('  [Sin resultados]')
        continue
    for _, row in resultados.iterrows():
        print(f"{int(row['Rank']):>3}  {row['Documento']:<80} {row['Similitud']:.6f}  {row['Similitud (%)']}")


CONSULTA: "amor pasión romance"
  1  Las cien mejores poesías (lí­ricas) de la lengua castellana.txt                  0.108368  10.84%
  2  Romancero gitano.txt                                                             0.098185  9.82%
  3  Novelas ejemplares.txt                                                           0.097383  9.74%
  4  Libro de poemas.txt                                                              0.092912  9.29%
  5  Novelas y teatro.txt                                                             0.087268  8.73%
  6  Historia de la lengua y literatura castellana, Tomo 1  $b Desde los orígenes hasta Carlos V.txt 0.082107  8.21%
  7  Comedias El remedio en la desdicha; El mejor alcalde, el rey.txt                 0.078996  7.90%
  8  Novelas ejemplares y amorosas.txt                                                0.071981  7.20%

CONSULTA: "misterio intriga secreto"
  1  El tesoro misterioso.txt                                                         0.133804  1